# Momentum One — GPU Edition (Bot 1.5)
### Everything's baked in. Just run Cell 1, then Cell 2.

**FIRST: turn on the GPU.** Menu -> **Runtime** -> **Change runtime type** -> **L4 GPU** (or **A100** for ~2× more) -> **Save**.


## Cell 1 — load the bot + connect your Drive (run once)
A window pops up to connect your **Google Drive** (click Allow) — that's how the best brain survives Colab shutting off.


In [ ]:
import torch, os, shutil
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE  <-  Runtime > Change runtime type > L4 GPU, then re-run')
from google.colab import drive
drive.mount('/content/drive')
!git clone -q https://github.com/monty313/the-truth.git
%cd the-truth
!pip -q install pyyaml >/dev/null 2>&1

# ---- Drive link (hardened, review 2026-07-20): safe to re-run; frozen brains always
# ---- reach Drive; known-bad (timid-era) brains are purged BY SERIAL NUMBER so training
# ---- never resumes a brain we've proven can't make money.
import hashlib
DR = '/content/drive/MyDrive/momentum_gpu'
os.makedirs(DR, exist_ok=True)
KNOWN_BAD = ['745bafe170f2', '461dd4bf4dbd']   # sha256[:12] serials of the proven-flat era (records were counterfeit too)
def sn(p):
    return hashlib.sha256(open(p,'rb').read()).hexdigest()[:12]
for f in list(os.listdir(DR)):
    p = os.path.join(DR, f)
    if f.endswith('.pt') and os.path.isfile(p) and sn(p) in KNOWN_BAD:
        os.remove(p); print('purged known-flat brain from Drive:', f)
gp = os.path.join(DR, 'gpu_progress.json')      # its streak bar came from the counterfeit metric
if os.path.exists(gp) and not os.path.exists(gp + '.pre_fix_backup'):
    os.rename(gp, gp + '.pre_fix_backup'); print('old record book set aside (pre-fix metric)')
CK = 'artifacts/checkpoints'
if not os.path.islink(CK):                           # first run on this VM
    for f in os.listdir(CK):
        s2, dst = os.path.join(CK, f), os.path.join(DR, f)
        if f.endswith('.pt') and (not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(s2)):
            shutil.copy(s2, dst); print('-> Drive:', f)
    shutil.rmtree(CK, ignore_errors=True)
    os.symlink(DR, CK)
print('checkpoints live on your Drive:', DR)
print('ready.')


## Cell 2 — START TRAINING
Everything is baked in: **60% of practice on 3% target / 3.5% risk**, 40% random for range,
**decides every 5 min**, checks its record **hourly**, **runs until Colab stops it**.

Just run this cell. **Re-run it any time to continue** from your best brain.


In [ ]:
!python scripts/gpu_train.py


## Cell 3 — how is it doing? (optional)
Best streak so far, and every saved record. `pass0012` = 12 cleared days in a row.


In [ ]:
!cat artifacts/checkpoints/gpu_progress.json
print('\n--- saved records ---')
!ls -1 artifacts/checkpoints/ 2>/dev/null | grep gpu_pass || echo 'no records yet'


## Cell 4 — the good brain (optional)
Best brain auto-saves to **MyDrive / momentum_gpu / gpu_best.pt**. This also copies it to your computer.


In [ ]:
from google.colab import files
files.download('artifacts/checkpoints/gpu_best.pt')


## Cell 5 — double-check on the REAL sim (optional, ~5 min)
Runs the saved brain on your exact simulator (not the fast copy) so you can trust it.


In [ ]:
!python scripts/gpu_validate.py --csv data/XAUUSD_curriculum_2026.csv --ckpt gpu_best
